# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

High-Performance Cloud GPU inference (`large-v3-turbo` default + 6 Model Hub + `ECAPA-TDNN` + `Gemini 2.5 Flash`).

---
### Quick Start:
1. Click **Runtime → Run all** (or `Ctrl + F9`).
2. The VoiceDiary interface will render below. Record or upload audio — transcription starts automatically!

In [ ]:
# 1. Install GPU Acceleration Libraries
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:
# 2. Launch VoiceDiary — Desktop UI Parity (Open Glass Workspace)
import os, time, tempfile, json, re, urllib.request
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# ─── Hardware Engine Detection ───
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"⚡ Hardware Acceleration: {gpu_name} ({compute_dtype.upper()})")

# ─── Model Hub (Lazy Cached, large-v3-turbo Pre-Warmed) ───
_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        print(f"  [Model Hub] Loading {name} into VRAM...")
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(
            name, device=device_type, compute_type=compute_dtype,
            num_workers=2, download_root="/content/models/whisper"
        )
    return _model_cache[name]

print("Pre-warming OpenAI Large-v3-Turbo...")
get_model("large-v3-turbo")
print("✓ Large-v3-Turbo active in VRAM.")

# ─── ECAPA-TDNN Embedder ───
print("Pre-warming ECAPA-TDNN Speaker Diarization...")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa",
    run_opts={"device": device_type}
)
print("✓ ECAPA-TDNN ready.")

# ─── Roman Urdu Rule-Based Transliteration ───
_roman = {
    'آپ':'aap','کیسے':'kaisay','ہیں':'hain','کیا':'kya','کر':'kar',
    'رہے':'rahay','ہو':'ho','میں':'main','ہوں':'hoon','یہ':'yeh',
    'وہ':'woh','نہیں':'nahi','ٹھیک':'theek','شکریہ':'shukriya',
    'سلام':'salam','بہت':'bohot','اچھا':'acha','سوال':'sawaal','جواب':'jawab',
}
def to_roman(t):
    return " ".join(_roman.get(re.sub(r'[\u064B-\u065F\u0670]','',w), w) for w in t.split())

# ─── Ultra-Fast GPU Audio Loader ───
def load_16k(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(0, keepdim=True)
        if sr != 16000:
            wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav.squeeze().numpy().astype(np.float32)
    except Exception:
        d, sr = sf.read(path)
        if d.ndim > 1: d = d.mean(1)
        d = d.astype(np.float32)
        if sr != 16000:
            n = int(len(d) * 16000 / sr)
            d = np.interp(np.linspace(0, len(d), n, endpoint=False), np.arange(len(d)), d).astype(np.float32)
        return d

# ─── Maps & Palettes ───
COLORS = ['#6366F1', '#10B981', '#F59E0B', '#EC4899', '#06B6D4', '#8B5CF6', '#F97316']
MODEL_MAP = {
    '⚡ Large-v3-Turbo (809M)': 'large-v3-turbo',
    '⚡ Base (74M)': 'base',
    '⚡ Tiny (39M)': 'tiny',
    '⚡ Small (244M)': 'small',
    '⚡ Medium (769M)': 'medium',
    '⚡ Distil-Whisper (756M)': 'distil-large-v3',
}
LANG_MAP = {
    '🌐 Bilingual (Urdu + English)': (None, False),
    '🇵🇰 Pure Urdu Script (اردو)': ('ur', False),
    '🇬🇧 English Only': ('en', False),
    '🔤 Roman Urdu (Latin)': ('ur', True),
}

# ─── Core Transcription Pipeline ───
def transcribe(audio_path, model_choice, lang_choice, thresh_pct, vad_ms):
    if not audio_path or not os.path.exists(audio_path):
        empty_tr = """<div class="empty-state">
            <svg width="44" height="44" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" opacity="0.4"><path d="M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z"/></svg>
            <p>Lecture transcript will stream here</p>
            <p class="text-secondary">Click the record button or upload audio to capture lecture speech</p>
        </div>"""
        empty_spk = """<div class="empty-state">
            <svg width="36" height="36" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" opacity="0.4"><path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/><path d="M19 10v2a7 7 0 0 1-14 0v-2"/><line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/></svg>
            <p>No speakers detected</p>
            <p class="text-secondary">Start recording to identify professor & students</p>
        </div>"""
        return empty_tr, empty_spk, "<div class='export-prompt'>Transcribe audio first to download formatted lecture notes.</div>", "", None, None, None

    t0 = time.time()
    data = load_16k(audio_path)
    dur = len(data) / 16000.0
    mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
    model = get_model(mkey)
    target_lang, is_roman = LANG_MAP.get(lang_choice, (None, False))
    thresh = float(thresh_pct) / 100.0

    segs, _ = model.transcribe(
        data, beam_size=1, best_of=1, temperature=0.0,
        language=target_lang, without_timestamps=False,
        vad_filter=True, vad_parameters=dict(min_silence_duration_ms=int(vad_ms))
    )

    profiles = {}
    nxt = 1
    html_items = []
    plain_lines = []
    srt_lines = []
    json_data = []
    idx = 1

    for seg in segs:
        txt = seg.text.strip()
        if not txt: continue
        if is_roman: txt = to_roman(txt)
        s0, s1 = seg.start, seg.end
        chunk = data[int(s0*16000):int(s1*16000)]
        spk = 1
        if len(chunk) >= 8000:
            try:
                with torch.inference_mode():
                    w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                    e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                    en = e / (np.linalg.norm(e) or 1.0)
                bid, bsim = None, -1.0
                for sid, embs in profiles.items():
                    ms_ = max(float(np.dot(en, x)) for x in embs) if embs else 0
                    if ms_ > bsim: bsim, bid = ms_, sid
                if bid and bsim >= thresh:
                    spk = bid
                    if len(profiles[spk]) < 50: profiles[spk].append(en)
                else:
                    spk = nxt; profiles[spk] = [en]; nxt += 1
            except: pass

        c = COLORS[(spk - 1) % len(COLORS)]
        ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
        urdu = any('\u0600' <= ch <= '\u06FF' for ch in txt)
        rtl_class = " rtl" if urdu else ""

        # Speech bubble matching desktop .transcript-item
        html_items.append(f"""
        <div class="transcript-item" style="border-left: 3px solid {c};">
            <div class="speaker-tag">
                <span class="speaker-dot" style="background: {c};"></span>
                <strong style="color: {c};">Speaker {spk}</strong>
                <span class="transcript-time">[{ts}]</span>
            </div>
            <div class="transcript-text{rtl_class}">{txt}</div>
        </div>""")

        plain_lines.append(f"[{ts}] Speaker {spk}: {txt}")

        def srt_t(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_lines.append(f"{idx}\n{srt_t(s0)} --> {srt_t(s1)}\n[Speaker {spk}]: {txt}\n")
        json_data.append({"speaker": f"Speaker {spk}", "id": spk, "start": round(s0,2), "end": round(s1,2), "time": ts, "text": txt})
        idx += 1

    elapsed = time.time() - t0
    speed_x = dur / max(0.01, elapsed)

    # Sidebar Speaker Cards
    sb = []
    for sid, embs in profiles.items():
        cc = COLORS[(sid-1) % len(COLORS)]
        sb.append(f"""
        <div class="speaker-card">
            <div class="speaker-avatar" style="background: {cc};">S{sid}</div>
            <div class="speaker-info">
                <div class="speaker-name">Speaker {sid}</div>
                <div class="speaker-meta">{len(embs)} voiceprint sample{'s' if len(embs) > 1 else ''}</div>
            </div>
        </div>""")
    if not sb:
        sb.append("""<div class="empty-state"><p>No speakers detected</p></div>""")

    # Stats pill
    stats_bar = f"""
    <div class="transcript-stats">
        <span>⚡ <strong>Engine:</strong> {mkey} · {gpu_name}</span>
        <span>⏱️ <strong>Audio:</strong> {dur:.1f}s → <strong>GPU Time:</strong> {elapsed:.1f}s (<strong>{speed_x:.1f}× real-time</strong>)</span>
    </div>"""

    transcript_html = f"<div class='transcript-list'>{''.join(html_items)}</div>" + stats_bar
    sidebar_html = "".join(sb)

    # Export generation
    files = {}
    for ext, content in [
        ('.md', "# VoiceDiary Classroom Notes\n\n" + "\n\n".join(plain_lines)),
        ('.txt', "\n".join(plain_lines)),
        ('.srt', "\n".join(srt_lines)),
        ('.json', json.dumps(json_data, indent=2, ensure_ascii=False))
    ]:
        f = tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False, encoding='utf-8', prefix='VoiceDiary_')
        f.write(content); f.close()
        files[ext] = f.name

    export_html = f"""
    <div class="export-bar">
        <a href="/file={files['.md']}" download class="btn-export">
            <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"/><polyline points="7 10 12 15 17 10"/><line x1="12" y1="15" x2="12" y2="3"/></svg>
            Markdown (.md)
        </a>
        <a href="/file={files['.txt']}" download class="btn-export">
            <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
            Text (.txt)
        </a>
        <a href="/file={files['.srt']}" download class="btn-export">
            <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2"/><line x1="7" y1="2" x2="7" y2="22"/><line x1="17" y1="2" x2="17" y2="22"/></svg>
            Subtitles (.srt)
        </a>
        <a href="/file={files['.json']}" download class="btn-export">
            <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>
            JSON (.json)
        </a>
    </div>"""

    return transcript_html, sidebar_html, export_html, "\n".join(plain_lines), files.get('.md'), files.get('.txt'), files.get('.srt')

# ─── Gemini Summary ───
def gemini_summary(text, key):
    if not text or not text.strip(): return "*No transcript available yet. Record or transcribe audio first.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY", "")
    if not api_key: return "⚠️ **Please enter your Gemini API Key in the Settings toolbar above to generate AI study notes.**"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    payload = {"contents":[{"parts":[{"text":f"You are VoiceDiary AI, an academic classroom note summarizer. Analyze this diarized lecture transcript and produce a rich, beautifully formatted study guide with:\n1. 🎯 Executive Lecture Summary\n2. 💡 Key Concepts & Definitions\n3. 📝 Exam Highlights & Key Takeaways\n4. ❓ Classroom Q&A Discussion Points\n\nTranscript:\n{text}"}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read())["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e: return f"❌ Error generating summary: {e}"

# ─── CSS: Open, Fluid, Un-Boxed Desktop Theme ───
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;600;700&display=swap');

/* Color Variables */
:root {
  --bg-base: #080C14;
  --bg-surface: #0F172A;
  --bg-surface-elevated: #1E293B;
  --bg-card: rgba(15, 23, 42, 0.65);
  --bg-glass: rgba(15, 23, 42, 0.45);
  --border-subtle: rgba(255, 255, 255, 0.08);
  --border-medium: rgba(255, 255, 255, 0.14);
  --border-focus: #6366F1;
  --text-primary: #F8FAFC;
  --text-secondary: #94A3B8;
  --text-muted: #64748B;
  --accent-primary: #6366F1;
  --accent-gradient: linear-gradient(135deg, #6366F1 0%, #8B5CF6 100%);
  --accent-glow: rgba(99, 102, 241, 0.25);
  --radius-sm: 6px;
  --radius-md: 10px;
  --radius-lg: 14px;
  --radius-xl: 20px;
  --radius-full: 9999px;
}

/* Global Reset & Canvas */
body, .gradio-container {
  background-color: var(--bg-base) !important;
  background-image: 
    radial-gradient(ellipse 80% 50% at 50% -20%, rgba(99, 102, 241, 0.15), transparent),
    radial-gradient(ellipse 60% 40% at 90% 90%, rgba(139, 92, 246, 0.08), transparent) !important;
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
  color: var(--text-primary) !important;
  max-width: 1560px !important;
  margin: 0 auto !important;
  padding: 12px 20px !important;
}

/* Eliminate ALL Gradio nested box chrome & borders */
.gradio-container .block,
.gradio-container .gr-panel,
.gradio-container .gr-box,
.gradio-container .gr-form,
.gradio-container .gr-group {
  background: transparent !important;
  border: none !important;
  box-shadow: none !important;
  padding: 0 !important;
  margin: 0 !important;
}

/* Top App Header */
.app-header {
  height: 64px;
  background: var(--bg-glass);
  backdrop-filter: blur(24px);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-xl);
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 0 24px;
  margin-bottom: 16px;
}
.header-left { display: flex; align-items: center; gap: 14px; }
.brand-logo-wrap {
  width: 40px; height: 40px; border-radius: var(--radius-md);
  background: var(--accent-gradient);
  display: flex; align-items: center; justify-content: center;
  box-shadow: 0 0 16px var(--accent-glow);
  color: #FFFFFF;
}
.app-title { font-size: 19px; font-weight: 800; color: #FFFFFF; letter-spacing: -0.02em; margin: 0; line-height: 1.2; }
.app-subtitle { font-size: 11px; font-weight: 500; color: var(--text-secondary); margin: 0; }

.hw-badge-pill {
  display: flex; align-items: center; gap: 8px;
  background: rgba(16, 185, 129, 0.1);
  border: 1px solid rgba(16, 185, 129, 0.3);
  padding: 6px 14px; border-radius: var(--radius-full);
  font-size: 11px; font-weight: 700; color: #10B981;
  font-family: 'Fira Code', monospace;
}
.hw-badge-dot {
  width: 7px; height: 7px; border-radius: 50%;
  background: #10B981; box-shadow: 0 0 8px rgba(16, 185, 129, 0.8);
}

/* Control Strip (Model + Lang + Settings in one horizontal ribbon) */
.control-ribbon {
  background: var(--bg-card);
  backdrop-filter: blur(16px);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-xl);
  padding: 12px 20px;
  margin-bottom: 16px;
  display: flex;
  align-items: center;
  gap: 16px;
  flex-wrap: wrap;
}

/* Left Sidebar - Speakers Panel (Continuous Glass) */
.sidebar-panel {
  background: var(--bg-card);
  backdrop-filter: blur(20px);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-xl);
  padding: 20px;
  min-height: 580px;
  display: flex;
  flex-direction: column;
}
.panel-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding-bottom: 14px;
  border-bottom: 1px solid var(--border-subtle);
  margin-bottom: 14px;
}
.panel-header h2 {
  font-size: 12px; font-weight: 800; color: var(--text-secondary);
  text-transform: uppercase; letter-spacing: 0.08em; margin: 0;
}
.count-badge {
  font-size: 11px; font-weight: 700; padding: 2px 10px;
  border-radius: var(--radius-full);
  background: rgba(99, 102, 241, 0.18); color: #818CF8;
}

.speaker-list { display: flex; flex-direction: column; gap: 10px; }
.speaker-card {
  display: flex; align-items: center; gap: 12px;
  padding: 12px 16px; border-radius: var(--radius-lg);
  background: rgba(255, 255, 255, 0.03);
  border: 1px solid var(--border-subtle);
  transition: 0.2s ease;
}
.speaker-card:hover {
  background: rgba(255, 255, 255, 0.07);
  border-color: var(--border-medium);
  transform: translateY(-1px);
}
.speaker-avatar {
  width: 36px; height: 36px; border-radius: var(--radius-full);
  display: flex; align-items: center; justify-content: center;
  font-weight: 800; font-size: 13px; color: #FFFFFF; flex-shrink: 0;
}
.speaker-info { flex: 1; min-width: 0; }
.speaker-name { font-size: 13px; font-weight: 600; color: var(--text-primary); }
.speaker-meta { font-size: 11px; color: var(--text-muted); margin-top: 2px; }

/* Right Workspace Panel (Spacious, Airy Canvas) */
.workspace-panel {
  background: var(--bg-card);
  backdrop-filter: blur(20px);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-xl);
  padding: 24px;
  min-height: 580px;
  display: flex;
  flex-direction: column;
}

/* Clean Mode Tabs */
.tabs { border: none !important; margin-bottom: 12px !important; }
.tab-nav { background: transparent !important; border-bottom: 1px solid var(--border-subtle) !important; gap: 8px !important; }
.tab-nav button {
  background: transparent !important;
  border: 1px solid transparent !important;
  border-bottom: none !important;
  border-radius: var(--radius-lg) var(--radius-lg) 0 0 !important;
  color: var(--text-secondary) !important;
  font-weight: 600 !important; font-size: 13px !important;
  padding: 10px 20px !important;
  transition: 0.2s ease !important;
}
.tab-nav button.selected {
  background: rgba(255, 255, 255, 0.05) !important;
  border-color: var(--border-subtle) !important;
  color: #FFFFFF !important;
}

/* Audio Widget Override (Minimalist, No Clunky Frame) */
.gr-audio {
  background: rgba(0, 0, 0, 0.25) !important;
  border: 1px solid var(--border-subtle) !important;
  border-radius: var(--radius-lg) !important;
  padding: 8px 12px !important;
}

/* Primary Transcribe Button */
.btn-primary-action {
  background: var(--accent-gradient) !important;
  color: #FFFFFF !important;
  font-weight: 700 !important;
  font-size: 14px !important;
  border: none !important;
  border-radius: var(--radius-lg) !important;
  padding: 12px 24px !important;
  cursor: pointer !important;
  box-shadow: 0 4px 18px rgba(99, 102, 241, 0.35) !important;
  transition: all 0.2s ease !important;
}
.btn-primary-action:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 6px 24px rgba(99, 102, 241, 0.5) !important;
}

/* Transcript Stream Viewport */
.transcript-canvas {
  flex: 1;
  background: rgba(0, 0, 0, 0.2);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 20px;
  min-height: 380px;
  max-height: 520px;
  overflow-y: auto;
  margin: 16px 0;
}
.transcript-list { display: flex; flex-direction: column; gap: 12px; }

/* Individual Speech Bubble Item (Desktop Exact Parity) */
.transcript-item {
  display: flex;
  flex-direction: column;
  gap: 6px;
  padding: 14px 18px;
  border-radius: var(--radius-lg);
  background: rgba(255, 255, 255, 0.03);
  border: 1px solid var(--border-subtle);
  transition: 0.15s ease;
}
.transcript-item:hover {
  background: rgba(255, 255, 255, 0.06);
  border-color: var(--border-medium);
}
.speaker-tag {
  display: flex;
  align-items: center;
  gap: 8px;
  font-size: 13px;
  font-weight: 700;
}
.speaker-dot {
  width: 8px; height: 8px; border-radius: 50%;
}
.transcript-time {
  font-size: 11px; font-weight: 500; color: var(--text-muted);
  font-family: 'Fira Code', monospace;
}
.transcript-text {
  font-size: 15px;
  line-height: 1.65;
  color: #E2E8F0;
  word-break: break-word;
}
.transcript-text.rtl {
  direction: rtl;
  text-align: right;
  font-family: 'Noto Nastaliq Urdu', 'Jameel Noori Nastaleeq', 'Segoe UI Historic', serif;
  font-size: 17px;
  line-height: 2.1;
  color: #F8FAFC;
}

/* Empty State Placeholder */
.empty-state {
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  text-align: center;
  padding: 60px 20px;
  color: var(--text-muted);
  gap: 10px;
}
.empty-state p { font-size: 15px; font-weight: 600; color: var(--text-secondary); margin: 0; }
.empty-state .text-secondary { font-size: 12px; color: var(--text-muted); margin: 0; }

/* Stats Bar */
.transcript-stats {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 12px 16px;
  margin-top: 12px;
  border-top: 1px solid var(--border-subtle);
  font-size: 11px;
  color: var(--text-secondary);
  font-family: 'Fira Code', monospace;
}

/* Export Buttons Bar */
.export-bar {
  display: flex;
  align-items: center;
  gap: 10px;
  flex-wrap: wrap;
  padding: 8px 0;
}
.btn-export {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  padding: 8px 16px;
  border-radius: var(--radius-full);
  background: rgba(255, 255, 255, 0.05);
  border: 1px solid var(--border-subtle);
  color: var(--text-primary);
  font-size: 12px;
  font-weight: 600;
  text-decoration: none;
  transition: 0.15s ease;
}
.btn-export:hover {
  background: rgba(99, 102, 241, 0.18);
  border-color: #818CF8;
  color: #FFFFFF;
}
.btn-export svg { color: var(--accent-primary); }

.export-prompt {
  font-size: 12px;
  color: var(--text-muted);
  padding: 6px 0;
}

/* Inputs & Dropdowns Overrides */
input, select, textarea, .gr-dropdown {
  background: var(--bg-surface-elevated) !important;
  border: 1px solid var(--border-subtle) !important;
  border-radius: var(--radius-md) !important;
  color: var(--text-primary) !important;
}
input:focus, select:focus {
  border-color: var(--border-focus) !important;
  box-shadow: 0 0 10px var(--accent-glow) !important;
}

/* Slider Overrides */
input[type=range] { accent-color: var(--accent-primary) !important; }
.gr-slider input[type=number] {
  background: var(--bg-surface-elevated) !important;
  border: 1px solid var(--border-subtle) !important;
  color: var(--text-primary) !important;
  border-radius: var(--radius-sm) !important;
}

/* Accordion for Settings / Tuning */
.gr-accordion {
  background: rgba(255, 255, 255, 0.02) !important;
  border: 1px solid var(--border-subtle) !important;
  border-radius: var(--radius-lg) !important;
  margin-top: 10px !important;
}

/* Hide raw file boxes & gradio footer */
.gr-file { display: none !important; }
footer { display: none !important; }
"""

with gr.Blocks(title="VoiceDiary — AI Classroom Lecture & Diarization Engine",
               css=CSS, theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    transcript_state = gr.State("")

    # ─── APP HEADER BAR ───
    gr.HTML(f"""
    <header class="app-header">
        <div class="header-left">
            <div class="brand-logo-wrap">
                <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2">
                    <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
                    <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
                    <line x1="12" y1="19" x2="12" y2="23"/>
                    <line x1="8" y1="23" x2="16" y2="23"/>
                </svg>
            </div>
            <div>
                <h1 class="app-title">VoiceDiary</h1>
                <p class="app-subtitle">AI Bilingual Lecture & Diarization Engine</p>
            </div>
        </div>
        <div class="hw-badge-pill">
            <span class="hw-badge-dot"></span>
            <span>{gpu_name} · Tensor Cores {compute_dtype.upper()}</span>
        </div>
    </header>""")

    # ─── TOP CONTROL RIBBON (Model + Lang + Tuning in horizontal flow) ───
    with gr.Row(elem_classes=["control-ribbon"]):
        with gr.Column(scale=4, min_width=240):
            model_dd = gr.Dropdown(
                choices=list(MODEL_MAP.keys()),
                value='⚡ Large-v3-Turbo (809M)',
                label='Active AI Model',
                show_label=True
            )
        with gr.Column(scale=4, min_width=240):
            lang_dd = gr.Dropdown(
                choices=list(LANG_MAP.keys()),
                value='🌐 Bilingual (Urdu + English)',
                label='Language Mode',
                show_label=True
            )
        with gr.Column(scale=4, min_width=240):
            gemini_key = gr.Textbox(
                placeholder='Paste Gemini API Key for AI Notes...',
                type='password',
                label='Gemini API Key (BYOK)',
                show_label=True
            )

    # ─── MAIN WORKSPACE (2 Column Open Split) ───
    with gr.Row():
        # LEFT SIDEBAR: SPEAKERS
        with gr.Column(scale=3, min_width=260, elem_classes=["sidebar-panel"]):
            gr.HTML("""
            <div class="panel-header">
                <h2>Speakers & Profiles</h2>
                <span class="count-badge">LIVE</span>
            </div>""")
            sidebar_out = gr.HTML(value="""
            <div class="empty-state">
                <svg width="36" height="36" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" opacity="0.4"><path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/><path d="M19 10v2a7 7 0 0 1-14 0v-2"/><line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/></svg>
                <p>No speakers detected</p>
                <p class="text-secondary">Start recording to identify speakers</p>
            </div>""")

            with gr.Accordion("⚙️ Engine Tuning", open=False):
                thresh_sl = gr.Slider(20, 70, 32, step=1, label='Diarization Sensitivity (%)')
                vad_sl = gr.Slider(150, 600, 280, step=10, label='VAD Silence Gap (ms)')

        # RIGHT MAIN CANVAS: TRANSCRIPTION & ACTIONS
        with gr.Column(scale=9, elem_classes=["workspace-panel"]):
            with gr.Tabs():
                with gr.TabItem("🎙️ Live Classroom Lecture"):
                    audio_mic = gr.Audio(sources=["microphone"], type="filepath", label="Classroom Microphone")
                with gr.TabItem("📁 Upload Audio File"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath", label="Upload Lecture (.wav .mp3 .m4a .flac)")

            with gr.Row():
                transcribe_btn = gr.Button("⚡ Transcribe & Diarize (GPU)", elem_classes=["btn-primary-action"])

            # Transcript Stream Canvas
            transcript_out = gr.HTML(
                value="""<div class="empty-state">
                    <svg width="44" height="44" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" opacity="0.4"><path d="M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z"/></svg>
                    <p>Lecture transcript will stream here</p>
                    <p class="text-secondary">Click the record button or upload audio to capture lecture speech</p>
                </div>""",
                elem_classes=["transcript-canvas"]
            )

            # Export Section
            with gr.Row():
                export_html_out = gr.HTML(value="<div class='export-prompt'>Transcribe audio first to download formatted lecture notes.</div>")
                # Hidden file endpoints for downloads
                with gr.Row(visible=False):
                    f_md = gr.File(); f_txt = gr.File(); f_srt = gr.File()

            # AI Study Notes Accordion
            with gr.Accordion("✨ AI Lecture Summary & Study Flashcards", open=False):
                ai_btn = gr.Button("Generate AI Study Notes (Gemini 2.5 Flash)", elem_classes=["btn-primary-action"])
                ai_out = gr.Markdown(value="*AI summary will be generated here once you click the button above.*")

    # ─── EVENT WIRING ───
    all_inputs = [audio_mic, audio_file, model_dd, lang_dd, thresh_sl, vad_sl]
    all_outputs = [transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]

    def run_from_mic(mic, _file, mod, lang, th, vad):
        return transcribe(mic, mod, lang, th, vad)
    def run_from_btn(mic, fpath, mod, lang, th, vad):
        return transcribe(mic if mic else fpath, mod, lang, th, vad)

    audio_mic.stop_recording(fn=run_from_mic, inputs=all_inputs, outputs=all_outputs)
    transcribe_btn.click(fn=run_from_btn, inputs=all_inputs, outputs=all_outputs)
    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
